In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

# اسم الموديل الجاهز المخصص لإشارات المرور (GTSRB)
model_name = "bazyl/gtsrb-model"

# تحميل الـ processor (اللي بيعمل preprocessing) والموديل نفسه
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

print("✅ Model loaded")
print("Number of classes:", model.config.num_labels)


In [ ]:
import torch
from PIL import Image

def predict_traffic_sign(img: Image.Image):
    """
    تاخد صورة PIL وترجع:
    - pred_id: رقم الكلاس
    - label: اسم الإشارة
    - confidence: نسبة الثقة
    """
    inputs = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)[0]
    pred_id = int(probs.argmax())
    confidence = float(probs[pred_id])
    label = model.config.id2label[pred_id]
    return pred_id, label, confidence


In [ ]:
from pathlib import Path
from IPython.display import display

images = sorted(path for path in Path("inputs").glob("*")
                if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})
if not images:
    raise FileNotFoundError("Add sample images to the inputs directory.")
for path in images:
    with Image.open(path) as source:
        img = source.convert("RGB")
    display(img)
    pred_id, label, confidence = predict_traffic_sign(img)
    print(f"{path.name}: {label} (class {pred_id}, confidence {confidence:.3f})")
